In [1]:
"""
OTC-based Observable Comparison Experiment
比較 SV, Noise Model, FakeMarrakesh, ibm_marrakesh (Baseline/ZNE/Twirling/ZNE+Twirling/QESEM)
Group 1 (target=+0.8): Z0, Z4, Z6, X5, X7
Group 2 (target=-0.8): Z1, Z5, Z7, X4, X6
"""

import numpy as np
import pandas as pd
import csv, time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_catalog import QiskitServerless, QiskitFunctionsCatalog


In [2]:
def build_qru_circuit(data_input, hidden_input, params):
    """
    data_input:   numpy array, shape (8,)
    hidden_input: numpy array, shape (8,), 初始為全零
    params:       numpy array, shape (131,)
    """

    data   = QuantumRegister(4, 'data')
    hidden = QuantumRegister(4, 'hidden')
    anc    = QuantumRegister(2, 'anc')
    qc = QuantumCircuit(data, hidden, anc)

    # ==================
    # 1. ENCODING
    # ==================
    # Data: params[0:16], 每 qubit 4 個參數
    qc.barrier(label='ENC')
    for i in range(4):
        rx_angle = data_input[2*i] * params[4*i]     + params[4*i + 1]
        ry_angle = data_input[2*i + 1] * params[4*i + 2] + params[4*i + 3]
        qc.rx(rx_angle, data[i])
        qc.ry(ry_angle, data[i])

    # Hidden: params[16:32]
    for i in range(4):
        rx_angle = hidden_input[2*i] * params[16 + 4*i]     + params[16 + 4*i + 1]
        ry_angle = hidden_input[2*i + 1] * params[16 + 4*i + 2] + params[16 + 4*i + 3]
        qc.rx(rx_angle, hidden[i])
        qc.ry(ry_angle, hidden[i])

    # 循環 CX on data: 0→1→2→3→0
    for i in range(4):
        qc.cx(data[i], data[(i+1) % 4])

    # 循環 CX on hidden: 4→5→6→7→4
    for i in range(4):
        qc.cx(hidden[i], hidden[(i+1) % 4])

    # ==================
    # 2. UPDATE & RESET
    # ==================
    qc.barrier(label='UPD')
    qc.cx(data[0], hidden[0])
    qc.cswap(hidden[0], hidden[1], anc[0])

    # Rot on hidden[0]: params[32:35]
    qc.rz(params[32], hidden[0])
    qc.ry(params[33], hidden[0])
    qc.rz(params[34], hidden[0])

    qc.cswap(hidden[0], hidden[3], anc[1])
    # ancilla 保持 |0⟩，不需 reset

    # ==================
    # 3. VARIATIONAL LAYERS（4 層）
    # ==================
    idx = 35
    for layer in range(4):
        qc.barrier(label=f'VAR{layer+1}')
        # Rot on all non-ancilla qubits
        for reg in [data, hidden]:
            for i in range(4):
                qc.rz(params[idx],     reg[i]); idx += 1
                qc.ry(params[idx],     reg[i]); idx += 1
                qc.rz(params[idx],     reg[i]); idx += 1

        # 循環 CX on all non-ancilla: data[0..3] → hidden[0..3] → back
        all_q = [data[i] for i in range(4)] + [hidden[i] for i in range(4)]
        for i in range(8):
            qc.cx(all_q[i], all_q[(i+1) % 8])

    return qc

In [3]:
# ── 設定 ──────────────────────────────────────────────────────
CALIB_CSV  = 'qru_otc_multi_obs_20260503_205659.csv'
OUT_CSV    = f'otc_observable_comparison_{time.strftime("%Y%m%d_%H%M%S")}.csv'
N_REPS     = 3      # 每組電路重複 3 次
N_SHOTS    = 4000

# ── Observable 分組 ───────────────────────────────────────────
# (qubit, pauli, obs_name, out_idx, group_id, target_val)
OBS_GROUPS = {
    0: {  # Group 1: target = +0.8
        'target': +0.8,
        'obs': [
            (0, 'Z', 'Z0',  0),
            (4, 'Z', 'Z4',  8),
            (6, 'Z', 'Z6', 12),
            (5, 'X', 'X5', 11),
            (7, 'X', 'X7', 15),
        ]
    },
    1: {  # Group 2: target = -0.8
        'target': -0.8,
        'obs': [
            (1, 'Z', 'Z1',  2),
            (5, 'Z', 'Z5', 10),
            (7, 'Z', 'Z7', 14),
            (4, 'X', 'X4',  9),
            (6, 'X', 'X6', 13),
        ]
    },
}

# ── OTC 資料庫讀取 ────────────────────────────────────────────
param_cols = [c for c in pd.read_csv(CALIB_CSV, nrows=0).columns if c.startswith('p')]
calib_df   = pd.read_csv(CALIB_CSV)

def get_params(group_id, target_val, rep_id):
    """從 OTC 資料庫取得電路參數"""
    obs_name = OBS_GROUPS[group_id]['obs'][0][2]  # 用第一個 obs 查詢
    mask = (
        (calib_df['group_id'] == group_id) &
        (calib_df['target_val'] == target_val) &
        (calib_df['rep_id'] == rep_id) &
        (calib_df['obs_name'] == obs_name)
    )
    rows = calib_df[mask]
    if len(rows) == 0:
        return None, None
    row     = rows.iloc[0]
    params  = row[param_cols].values.astype(float)
    sv_vals = {}
    for _, r in calib_df[(calib_df['group_id']==group_id) &
                          (calib_df['target_val']==target_val) &
                          (calib_df['rep_id']==rep_id)].iterrows():
        sv_vals[r['obs_name']] = r['sv_val']
    return params, sv_vals

# ── Observable 建構 ───────────────────────────────────────────
def make_obs_list(group_id, n=10):
    obs_list = []
    for q, p, name, _ in OBS_GROUPS[group_id]['obs']:
        s = ['I'] * n
        s[n-1-q] = p
        obs_list.append(SparsePauliOp(''.join(s)))
    return obs_list


In [4]:
# ── 寫 CSV header ─────────────────────────────────────────────
with open(OUT_CSV, 'w', newline='') as f:
    csv.writer(f).writerow([
        'method', 'group_id', 'obs_name', 'target_val',
        'rep_id', 'sv_val', 'measured_val'
    ])

def write_results(method, group_id, obs_names, target_val, rep_id, sv_vals, evs):
    with open(OUT_CSV, 'a', newline='') as f:
        writer = csv.writer(f)
        for i, obs_name in enumerate(obs_names):
            writer.writerow([
                method, group_id, obs_name, target_val,
                rep_id, sv_vals.get(obs_name, float('nan')),
                float(evs[i])
            ])


In [5]:
# ── Method 1: Statevector ─────────────────────────────────────
print("=== Running Statevector ===")
sv_estimator = StatevectorEstimator()

for grp_id, grp in OBS_GROUPS.items():
    target_val = grp['target']
    obs_names  = [o[2] for o in grp['obs']]
    obs_list   = make_obs_list(grp_id)

    for rep in range(N_REPS):
        params, sv_vals = get_params(grp_id, target_val, rep)
        if params is None:
            continue
        qc   = build_qru_circuit(np.zeros(8), np.zeros(8), params)
        evs  = sv_estimator.run([(qc, obs_list)]).result()[0].data.evs
        write_results('statevector', grp_id, obs_names, target_val, rep, sv_vals, evs)
        print(f"  SV grp={grp_id} rep={rep}: {[f'{v:.3f}' for v in evs]}")

# ── Method 2 & 3: Noise Model & FakeMarrakesh ─────────────────
for method_name, backend_obj in [
    ('noise_model',   AerSimulator(noise_model=NoiseModel.from_backend(FakeMarrakesh()))),
    ('fake_marrakesh', AerSimulator.from_backend(FakeMarrakesh())),
]:
    print(f"\n=== Running {method_name} ===")
    pm = generate_preset_pass_manager(backend=backend_obj, optimization_level=1)

    for grp_id, grp in OBS_GROUPS.items():
        target_val = grp['target']
        obs_names  = [o[2] for o in grp['obs']]
        obs_list   = make_obs_list(grp_id)

        for rep in range(N_REPS):
            params, sv_vals = get_params(grp_id, target_val, rep)
            if params is None:
                continue
            qc      = build_qru_circuit(np.zeros(8), np.zeros(8), params)
            isa_qc  = pm.run(qc)
            isa_obs = [o.apply_layout(isa_qc.layout) for o in obs_list]

            est = Estimator(mode=backend_obj)
            est.options.default_shots = N_SHOTS
            evs = est.run([(isa_qc, isa_obs)]).result()[0].data.evs
            write_results(method_name, grp_id, obs_names, target_val, rep, sv_vals, evs)
            print(f"  {method_name} grp={grp_id} rep={rep}: {[f'{v:.3f}' for v in evs]}")


=== Running Statevector ===
  SV grp=0 rep=0: ['0.800', '0.800', '0.799', '0.800', '0.800']
  SV grp=0 rep=1: ['0.800', '0.800', '0.799', '0.800', '0.801']
  SV grp=0 rep=2: ['0.801', '0.801', '0.800', '0.800', '0.800']
  SV grp=1 rep=0: ['-0.800', '-0.789', '-0.790', '-0.796', '-0.791']
  SV grp=1 rep=1: ['-0.800', '-0.800', '-0.800', '-0.799', '-0.801']
  SV grp=1 rep=2: ['-0.799', '-0.799', '-0.800', '-0.801', '-0.800']

=== Running noise_model ===
  noise_model grp=0 rep=0: ['0.667', '0.668', '0.679', '0.727', '0.682']
  noise_model grp=0 rep=1: ['0.679', '0.672', '0.680', '0.711', '0.714']
  noise_model grp=0 rep=2: ['0.720', '0.675', '0.704', '0.695', '0.767']
  noise_model grp=1 rep=0: ['-0.741', '-0.694', '-0.690', '-0.706', '-0.704']
  noise_model grp=1 rep=1: ['-0.719', '-0.694', '-0.682', '-0.739', '-0.803']
  noise_model grp=1 rep=2: ['-0.699', '-0.701', '-0.690', '-0.698', '-0.737']

=== Running fake_marrakesh ===
  fake_marrakesh grp=0 rep=0: ['0.475', '0.445', '0.458', '

In [ ]:
# ── Methods 4-8: Real Backend ─────────────────────────────────
print("\n=== Running Real Backend (ibm_marrakesh) ===")
service = QiskitRuntimeService(name="NTU")
backend = service.backend('ibm_marrakesh', instance="<YOUR_INSTANCE_NAME>")

pm_real = generate_preset_pass_manager(backend=backend, optimization_level=1)

print(backend.options.shots)

In [ ]:
# ── Methods 4-8: Fake Backend ─────────────────────────────────
print("\n=== Running Fake Backend (FakeMarrakesh) ===")
backend = AerSimulator.from_backend(FakeMarrakesh())

pm_real = generate_preset_pass_manager(backend=backend, optimization_level=1)

In [6]:
# 預先 transpile 所有電路
circuits = {}  # (grp_id, rep) -> (isa_qc, isa_obs, sv_vals, obs_names)
for grp_id, grp in OBS_GROUPS.items():
    target_val = grp['target']
    obs_names  = [o[2] for o in grp['obs']]
    obs_list   = make_obs_list(grp_id)
    for rep in range(N_REPS):
        params, sv_vals = get_params(grp_id, target_val, rep)
        if params is None:
            continue
        qc      = build_qru_circuit(np.zeros(8), np.zeros(8), params)
        isa_qc  = pm_real.run(qc)
        isa_obs = [o.apply_layout(isa_qc.layout) for o in obs_list]
        circuits[(grp_id, rep)] = (isa_qc, isa_obs, sv_vals, obs_names, target_val)

# 各方法的 options
METHODS_REAL = {
    'baseline': EstimatorOptions(),
    'zne': EstimatorOptions(),
    'twirling': EstimatorOptions(),
    'zne_twirling': EstimatorOptions(),
}
METHODS_REAL['zne'].resilience_level = 2
METHODS_REAL['twirling'].twirling.enable_gates = True
METHODS_REAL['twirling'].twirling.num_randomizations = 300
METHODS_REAL['zne_twirling'].resilience_level = 2
METHODS_REAL['zne_twirling'].twirling.enable_gates = True
METHODS_REAL['zne_twirling'].twirling.num_randomizations = 300

st = time.time()
with Session(backend=backend) as session:
    # # Methods 4-7: Baseline, ZNE, Twirling, ZNE+Twirling
    # for method_name, options in METHODS_REAL.items():
    #     print(f"\n  --- {method_name} ---")
    #     estimator = Estimator(mode=session, options=options)
    #     for (grp_id, rep), (isa_qc, isa_obs, sv_vals, obs_names, target_val) in circuits.items():
    #         evs = estimator.run([(isa_qc, isa_obs)]).result()[0].data.evs
    #         write_results(method_name, grp_id, obs_names, target_val, rep, sv_vals, evs)
    #         elapsed = time.time() - st
    #         print(f"    grp={grp_id} rep={rep} ({elapsed/60:.1f}min): {[f'{v:.3f}' for v in evs]}")


    # Method 8: QESEM
    print(f"\n  --- QESEM ---")
    try:
        catalog = QiskitFunctionsCatalog(
            channel="ibm_quantum_platform",
            instance="<YOUR_CRN_INSTANCE>"
        )
        qesem_fn = catalog.load('qedma/qesem')

        # QESEM 只跑 rep=0
        for grp_id in OBS_GROUPS.keys():
            key = (grp_id, 0)
            if key not in circuits:
                continue
            isa_qc, isa_obs, sv_vals, obs_names, target_val = circuits[key]
        
            job = qesem_fn.run(
                pubs=[(isa_qc, isa_obs)],
                backend_name='ibm_marrakesh',
                instance="<YOUR_CRN_INSTANCE>",
            )
            result = job.result()
            evs = [result[0].data.evs[i] for i in range(len(isa_obs))]
            write_results('qesem', grp_id, obs_names, target_val, 0, sv_vals, evs)
            elapsed = time.time() - st
            print(f"    grp={grp_id} rep=0 ({elapsed/60:.1f}min): {[f'{v:.3f}' for v in evs]}")
            
    except Exception as e:
        print(f"    QESEM error: {e}")

print(f"\n完成！儲存至 {OUT_CSV}")
print(f"總時間: {(time.time()-st)/60:.1f} 分鐘")




  --- QESEM ---
    QESEM error: 
| Message: Connection error. Make sure configuration (host and auth details) is correct.
| Code: AUTH1001
| Details: (MaxRetryError('HTTPSConnectionPool(host=\'qiskit-serverless.quantum.ibm.com\', port=443): Max retries exceeded with url: /api/v1/jobs/b98bc021-86b1-4453-bdfa-fafbd075424f/?with_result=false (Caused by NameResolutionError("HTTPSConnection(host=\'qiskit-serverless.quantum.ibm.com\', port=443): Failed to resolve \'qiskit-serverless.quantum.ibm.com\' ([Errno -3] Temporary failure in name resolution)"))'),)

完成！儲存至 otc_observable_comparison_20260504_215810.csv
總時間: 101.3 分鐘


In [ ]:
# ── 快速分析 ──────────────────────────────────────────────────
print("\n=== 各方法的平均期望值 vs SV 目標 ===")
result_df = pd.read_csv(OUT_CSV)
methods   = result_df['method'].unique()

obs_order = ['Z0','Z1','Z4','X4','Z5','X5','Z6','X6','Z7','X7']
targets   = {'Z0':0.8,'Z4':0.8,'Z6':0.8,'X5':0.8,'X7':0.8,
             'Z1':-0.8,'Z5':-0.8,'Z7':-0.8,'X4':-0.8,'X6':-0.8}

print(f"{'obs':<6} {'target':>8} " + "".join(f"{m[:8]:>10}" for m in methods))
print("-" * (14 + 10*len(methods)))
for obs in obs_order:
    row = f"{obs:<6} {targets[obs]:>8.1f} "
    for m in methods:
        sub = result_df[(result_df['obs_name']==obs) & (result_df['method']==m)]
        val = sub['measured_val'].mean() if len(sub) > 0 else float('nan')
        row += f"{val:>10.4f}"
    print(row)

In [8]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(
    channel="ibm_quantum_platform",
    instance="<YOUR_CRN_INSTANCE>"
)

# 直接從 catalog 取 job
job = catalog.job('b98bc021-86b1-4453-bdfa-fafbd075424f')
print(job.status())
result = job.result()
print(result)

DONE
PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(5,), dtype=float64>), stds=np.ndarray(<shape=(5,), dtype=float64>), shape=(5,)), metadata={'gate_fidelities': {'CZ': 0.9952606904166358, 'ID1Q': 0.9981410142178174}, 'total_shots': 8795858, 'mitigation_shots': 3536658, 'transpiled_circs': [{'circuit': 'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[72] c0;\nqubit[72] q0;\nrx(-pi) q0[55];\nrx(1.7172414303556671) q0[54];\nrx(1.6132446608741322) q0[53];\nrx(0.19154055640744794) q0[52];\nrx(-3.0648477889300527) q0[51];\nrx(1.5160571014909845) q0[50];\nrx(1.5701347589467485) q0[49];\nrx(1.6750431370979406) q0[58];\nrx(3*pi/4) q0[71];\nrx(-pi) q0[70];\nrz(0.8908416533118206) q0[55];\nrz(1.8065781830057652) q0[54];\nrz(-pi/2) q0[53];\nrz(pi/2) q0[52];\nrz(pi/2) q0[51];\nrz(-1.528360699377025) q0[50];\nrz(-1.511818565824358) q0[49];\nrz(-3.0262167428120446) q0[58];\nrz(0) q0[71];\nrz(-1.574730995150917) q0[70];\nrx(-3.0213632448044816) q0[55];\nrx(pi/2) q0[54];\nrx(-1.90052142648

In [9]:
import csv

# 手動寫入 Group 1 QESEM 結果
obs_names_grp1 = ['Z0', 'Z4', 'Z6', 'X5', 'X7']
sv_vals_grp1   = {'Z0': 0.800, 'Z4': 0.800, 'Z6': 0.800, 'X5': 0.800, 'X7': 0.800}

qesem_vals = {
    'Z0': 0.8200,
    'Z4': 0.8172,
    'Z6': 0.7930,
    'X5': 0.8074,
    'X7': 0.7740,
}

with open('otc_observable_comparison_20260504_205308.csv', 'a', newline='') as f:
    writer = csv.writer(f)
    for obs_name in obs_names_grp1:
        writer.writerow([
            'qesem', 0, obs_name, 0.8, 0,
            sv_vals_grp1[obs_name],
            qesem_vals[obs_name]
        ])

print("Group 1 QESEM 結果已寫入 CSV")

Group 1 QESEM 結果已寫入 CSV
